# Mixed tetrahedral and prismatic Darcy

Explicit normal-degree-one spaces: tetrahedral18/P1, enriched32/P2, and prism27/W11. Pressure and flux use their physical volume norms. The well campaign partitions the same faceted domain; it does not identify the article's historical meshes.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm import AffineMixedMesh, HDiv3DFamily

assert [
    HDiv3DFamily(k, p).local_size for k, p in [("tetrahedron", 1), ("tetrahedron", 2), ("prism", 1)]
] == [18, 32, 27]


## Declare the physical variational forms

The application defines local operators with the shared element, integration
and oriented face kernels and supplies the actual A/B/C blocks to the generic
assembler. For primal pressure the form is (K grad(p),grad(v)); mixed coordinates
use the inverse-permeability flux mass, divergence and normal-moment constraint.

$$
\begin{aligned}
A_Tu_T+B_T\lambda&=f_T,\\
C_Tu_T+D_T\lambda&=g_T, & C_T&=-B_T^{\mathsf T}, & D_T&=0.
\end{aligned}
$$

The local constant pressure and its physical integral moment retain the executed
nodal/modal basis. The separate global `Equation` supplies the signed pressure
boundary functional; pure Neumann data add a physical pressure integral constraint.
The flux trace follows fixed macroface normals. Mixed flux coordinates retain
their H(div) normal/interior moments and Piola transform; a primal gradient
field is evaluated independently. Generic assembly/solve and physical coefficient
interpretation are separate operations.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.mixed_darcy_3d import define_hdiv_darcy, recover_hdiv_darcy


def pressure(x):
    """Affine exact pressure with nonhomogeneous boundary values."""
    return 1 + x[:, 0] + 2 * x[:, 1] + 3 * x[:, 2]


with threadpool_limits(1):
    mesh = AffineMixedMesh.unit_cube(kind="prism")
    solution_definition = define_hdiv_darcy(mesh, local_refinement=1, dirichlet=pressure)
    solution_system = assemble(solution_definition.problem)
    solution = recover_hdiv_darcy(
        solution_definition,
        solution_system,
        solution_system.solve(
            constraints=pressure_constraints(solution_definition, solution_system)
        ),
    )
    error = solution.errors(pressure, [-1.0, -2.0, -3.0])
assert max(error.values()) < 1e-10
assert max((np.max(abs(v)) for v in solution.equilibrium_residuals())) < 1e-10
error


In [ ]:
folder = ROOT / "examples/results/mixed-well-geometries"
records = [json.loads(path.read_text()) for path in sorted(folder.glob("*-fine*-macro*.json"))]
assert len(records) >= 18
for row in records:
    assert row["archive_schema"] == 2
    assert hashlib.sha256((folder / row["archive"]).read_bytes()).hexdigest() == row["sha256"]
    with np.load(folder / row["archive"]) as archive:
        basis = archive["flux_basis_coefficients"]
        assert hashlib.sha256(basis.tobytes()).hexdigest() == row["flux_basis_sha256"]
    assert row["physical_block_residual"] < 1e-10
    assert row["fine_pressure_moment_defect"] < 1e-10
[(r["kind"], r["pressure_degree"], r["fine_factor"], r["macro_factor"],
  r["relative_errors"]["pressure_l2"], r["relative_errors"]["flux_l2"]) for r in records]

Exact and numerical field maps use identical top-face centroid samples, with no smoothing. Flux components and differences use zero-centered scales. The actual macro edges are drawn on all maps. Local resolution, skeletal resolution and pressure enrichment are separate studies. Flux components use a symmetric logarithmic scale with a linear core equal to 1% of the full displayed absolute limit; pressure remains linear. No field values are clipped.

The component comparisons show both the coarse skeletal approximation and its complete fine-face classical limit. Their volume errors distinguish skeletal restriction from fine-grid discretization. Archived flux coordinates are replayed with their stored reference-basis matrix; the record verifies both the field archive and the basis digest.

In [ ]:
for name in ["geometry", "convergence", "prism-p1-trace-comparison", "tetrahedron-p1-trace-comparison", "tetrahedron-p2-trace-comparison", "prism-p1-fields", "tetrahedron-p1-fields", "tetrahedron-p2-fields"]:
    display(Image(filename=str(ROOT / "docs/figures/mixed-well-geometries" / f"{name}.png")))